# Prepping Data Challenge - Week 06

## Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Import Data

In [2]:
salary_data = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_6_Input.csv')

In [4]:
salary_data.head()

,StaffID,1,2,3,4,5,6,7,8,9,10,11,12
0,1533,2398.0,2421.98,2446.20,2446.20,2495.12,2495.12,2495.12,2495.12,2545.03,2621.38,2621.38,2621.38
1,1339,7304.0,7523.12,7673.58,7673.58,7750.32,7827.82,8062.66,8304.54,8470.63,8555.33,8555.33,8726.44
2,2291,8240.0,8404.80,8572.90,8744.35,8831.80,9096.75,9278.69,9464.26,9464.26,9464.26,9558.90,9558.90
3,2038,3908.0,3986.16,3986.16,4026.02,4066.28,4188.27,4313.92,4443.34,4487.77,4622.40,4668.63,4715.31
4,2810,3988.0,4107.64,4148.72,4190.20,4274.01,4316.75,4316.75,4359.92,4490.71,4535.62,4671.69,4718.41


In [5]:
salary_data.shape

(999, 13)

## Challenges

### 1. Deduplicate by taking the latest row

In [7]:
current_salary = salary_data.copy().drop_duplicates(subset='StaffID',keep='last')

In [8]:
current_salary.shape

(803, 13)

Now that the `StaffId` is unique, set it as the index

In [10]:
current_salary = current_salary.set_index('StaffID')

### 2. Obtain annual salary

In [15]:
current_salary['Annual Salary'] = current_salary.T.sum()

In [19]:
current_salary = current_salary.drop([str(x) for x in[1,2,3,4,5,6,7,8,9,10,11,12]], axis=1)

### 3. Obtain highest Tax Bracket

Tax brackets are as follows:
- 'Personal Allowance': 0 - 12750 : 0%
- 'Basic Rate': 12751 - 50270 : 20%
- 'Higher Rate': 50271 - 125140 : 40%
- 'Additional Rate' : 125141 - inf : 45% 

In [21]:
def get_tax_bracket(salary):
    if salary <= 12750:
        rate = 0
    elif salary <= 50270:
        rate = 20
    elif salary <= 125140:
        rate = 40
    else:
        rate = 45
    return f"{rate}% rate"

In [22]:
current_salary['Highest Tax Bracket'] = current_salary['Annual Salary'].apply(get_tax_bracket)

In [23]:
current_salary

,Annual Salary,Highest Tax Bracket
StaffID,,
1339,96427.35,40% rate
2038,51412.26,40% rate
2810,52118.42,40% rate
1001,166864.48,45% rate
2943,34013.81,20% rate
...,...,...
2959,116412.02,40% rate
1467,26353.72,20% rate
2582,71088.71,40% rate


### 4. For each tax bracket, calculate the taxes paid

In [26]:
tax_brackets = {0.45 : {'lower_limit' : 125140, 'upper_limit' : 999999999}, 
                0.4 : {'lower_limit' : 50270, 'upper_limit' :125140},
                0.2 : {'lower_limit' : 12750, 'upper_limit' : 50270}}


def get_tax_paid(salary, tax_rate):
    # First thing: Check if salary too low to qualify
    if salary <= tax_brackets[tax_rate]['lower_limit']:
        return 0

    
    # For the highest tax bracket, do not apply upper limit
    if tax_rate == 0.45:
        return (salary - tax_brackets[tax_rate]['lower_limit']) * tax_rate

    # For all others, determine if salary this is the highest tax bracket. Calculate tax accordingly
    else:
        if salary > tax_brackets[tax_rate]['upper_limit']:
            return (tax_brackets[tax_rate]['upper_limit'] - tax_brackets[tax_rate]['lower_limit']) * tax_rate

        else:
            return (salary - tax_brackets[tax_rate]['lower_limit']) * tax_rate

In [28]:
# Use for loop to save one line of code
for rate in [0.2, 0.4, 0.45]:
    current_salary[f'{int(rate * 100)}% rate Tax paid'] = current_salary['Annual Salary'].apply(lambda x: get_tax_paid(x, rate))

In [29]:
current_salary

,Annual Salary,Highest Tax Bracket,20% rate Tax paid,40% rate Tax paid,45% rate Tax paid
StaffID,,,,,
1339,96427.35,40% rate,7504.000,18462.940,0.000
2038,51412.26,40% rate,7504.000,456.904,0.000
2810,52118.42,40% rate,7504.000,739.368,0.000
1001,166864.48,45% rate,7504.000,29948.000,18776.016
2943,34013.81,20% rate,4252.762,0.000,0.000
...,...,...,...,...,...
2959,116412.02,40% rate,7504.000,26456.808,0.000
1467,26353.72,20% rate,2720.744,0.000,0.000
2582,71088.71,40% rate,7504.000,8327.484,0.000


## Save results

In [30]:
current_salary.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/06_staff_taxes.csv', index_label='StaffID')